# S2 Pair Panel (v2 · 1D)

Builds the point-in-time S&P name panel and pair-panel cache for the v2 hypothesis ladder.
Not a hypothesis — freeze cuts and screen once, then filter in H-001+.

## Two passes

| Pass | `BOOK_SCOPE` | When | Writes |
|------|--------------|------|--------|
| A | `freeze_only` | Before any hyp | cuts, name panel, candidates, freeze pair panel |
| B | `freeze_plus_rotate` | After H-001 + H-002 (and preferably H-004) | rebalance schedules + rebuilt pair-panel union |

**Timing:** signal / decision at close `t`; fill at open `t+1`. Hedge / z / ADF / half-life use closes only.

**Filenames:** no `pit`, no universe letter (`s2_candidates_1d.csv`, `s2_panel_1d_train.parquet`, …).

The config cell prints the active next-step from `BOOK_SCOPE`:
- Pass A (`freeze_only`): Next: run H-001.
- Pass B (`freeze_plus_rotate`): Next: run H-005.


## 0. Imports & Config


In [ ]:
import json
import os
import sys
from itertools import combinations

import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import s2_artifacts_dir, s2_data_dir
from backtest.s2_coint.rotation import (
    quarter_end_rebalance_dates,
    semiannual_rebalance_dates,
)
from data.ingestion.equity_fetcher import fetch_ohlcv
from data.ingestion.equity_gics import fetch_yahoo_gics
from data.ingestion.sp500_universe import (
    constituents_on_or_before,
    load_sp500_snapshots,
)
from data.processing.s2_coint_store import build_pair_panel, screen_pair_cointegration
from strategies.s2_coint.book import select_book

NOTEBOOK_DIR = os.path.join(ROOT, "01_data", "data_files", "s2_coint", "v2")
os.makedirs(NOTEBOOK_DIR, exist_ok=True)
STAR_PATH = os.path.join(s2_artifacts_dir(version="v2", root=ROOT), "s2_star_stack.json")
GICS_CACHE = os.path.join(NOTEBOOK_DIR, "s2_yahoo_gics.parquet")

# "freeze_only" = Pass A (before hyps); "freeze_plus_rotate" = Pass B (after POOL/FDR).
BOOK_SCOPE = "freeze_only"

BAR = "1d"
FETCH_START = "1995-01-01"
END_DATE = None
ADV_FLOOR_USD = 5_000_000.0
ADV_WINDOW = 20
COINT_PVALUE_THRESHOLD = 0.05
OLS_WINDOW = 252
Z_WINDOW = 60
HL_WINDOW = 252
PANEL_UNION_BUFFER = 80  # enough for N=50 under both granularities
LOOKBACKS = (252, 504)

if BOOK_SCOPE not in {"freeze_only", "freeze_plus_rotate"}:
    raise ValueError("BOOK_SCOPE must be 'freeze_only' or 'freeze_plus_rotate'")

NEXT = (
    "Next: run H-001."
    if BOOK_SCOPE == "freeze_only"
    else "Next: run H-005."
)
print("BOOK_SCOPE", BOOK_SCOPE)
print(NEXT)
print("NOTEBOOK_DIR", NOTEBOOK_DIR)


## 1. PIT S&P membership union


In [ ]:
snapshots = load_sp500_snapshots()
snap = snapshots.loc[snapshots["date"] >= pd.Timestamp(FETCH_START)].copy()
if snap.empty:
    raise ValueError(f"no S&P snapshots on/after {FETCH_START}")

union: set[str] = set()
membership_rows: list[dict] = []
for _, row in snap.iterrows():
    tickers = [t.strip().upper() for t in str(row["tickers"]).split(",") if t.strip()]
    asof = pd.Timestamp(row["date"]).normalize()
    for t in tickers:
        union.add(t)
        membership_rows.append({"date": asof, "ticker": t, "in_index": True})

membership = pd.DataFrame(membership_rows)
print(f"snapshots={len(snap)}  union_names={len(union)}  "
      f"span={snap['date'].min().date()} → {snap['date'].max().date()}")


## 2. Fetch OHLCV + Yahoo drop-rate


In [ ]:
ohlc_by_ticker: dict[str, pd.DataFrame] = {}
closes: dict[str, pd.Series] = {}
dropped: list[str] = []

for i, ticker in enumerate(sorted(union)):
    raw = fetch_ohlcv(
        ticker,
        FETCH_START,
        END_DATE,
        auto_adjust=False,
        interval="1d",
    )
    if raw is None or raw.empty or "close" not in raw.columns:
        dropped.append(ticker)
        continue
    frame = raw.copy()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame = frame.sort_values("date").drop_duplicates("date")
    frame = frame.set_index("date")[["open", "high", "low", "close", "volume"]].astype(float)
    if frame["close"].notna().sum() < 60:
        dropped.append(ticker)
        continue
    ohlc_by_ticker[ticker] = frame
    closes[ticker] = frame["close"]
    if (i + 1) % 50 == 0:
        print(f"fetched {i+1}/{len(union)}")

n_union = len(union)
n_ok = len(ohlc_by_ticker)
drop_rate = 1.0 - (n_ok / n_union if n_union else float("nan"))
print(f"Yahoo drop-rate: {drop_rate:.2%}  ({n_union - n_ok}/{n_union} names)")
print("dropped sample:", dropped[:20], ("..." if len(dropped) > 20 else ""))
DROP_RATE = float(drop_rate)
DROPPED_TICKERS = list(dropped)


## 3. Yahoo GICS coverage


In [ ]:
gics = fetch_yahoo_gics(sorted(ohlc_by_ticker), cache_path=GICS_CACHE)
gics["has_gics"] = (gics["sector"] != "") & (gics["industry"] != "")
n_px = len(gics)
n_gics = int(gics["has_gics"].sum())
gics_miss_rate = 1.0 - (n_gics / n_px if n_px else float("nan"))
print(f"GICS missing among priced names: {gics_miss_rate:.2%}  ({n_px - n_gics}/{n_px})")
gics_ok = set(gics.loc[gics["has_gics"], "ticker"].astype(str))
sector_of = dict(zip(gics["ticker"].astype(str), gics["sector"].astype(str)))
industry_of = dict(zip(gics["ticker"].astype(str), gics["industry"].astype(str)))
GICS_MISS_RATE = float(gics_miss_rate)


## 4. ADV liquidity floor


In [ ]:
def adv_at(frame: pd.DataFrame, asof: pd.Timestamp, window: int = ADV_WINDOW) -> float:
    sub = frame.loc[frame.index <= asof].tail(window)
    if len(sub) < max(5, window // 2):
        return float("nan")
    dollar = (sub["close"] * sub["volume"]).astype(float)
    return float(dollar.mean())


# Provisional cut date ≈ 70th pct of the calendar of any liquid name; refined in §5.
probe_dates = sorted({d for fr in ohlc_by_ticker.values() for d in fr.index})
if not probe_dates:
    raise RuntimeError("no OHLCV dates")
probe_cut = pd.Timestamp(probe_dates[int(0.70 * (len(probe_dates) - 1))]).normalize()

adv_ok: set[str] = set()
adv_dropped: list[str] = []
for t in sorted(gics_ok):
    adv = adv_at(ohlc_by_ticker[t], probe_cut)
    if np.isfinite(adv) and adv >= ADV_FLOOR_USD:
        adv_ok.add(t)
    else:
        adv_dropped.append(t)

adv_drop_rate = len(adv_dropped) / max(len(gics_ok), 1)
print(f"ADV floor ${ADV_FLOOR_USD:,.0f} at {probe_cut.date()}: "
      f"drop {adv_drop_rate:.2%}  ({len(adv_dropped)}/{len(gics_ok)})")
print(f"tradable after GICS+ADV: {len(adv_ok)}")
ADV_DROP_RATE = float(adv_drop_rate)
TRADABLE = sorted(adv_ok)


## 5. Freeze research cuts (`s2_cuts.json`)


In [ ]:
# Unique trading dates across tradable names.
all_dates = sorted({d for t in TRADABLE for d in ohlc_by_ticker[t].index})
all_dates = pd.DatetimeIndex(all_dates)
if len(all_dates) < 100:
    raise RuntimeError(f"too few unique dates: {len(all_dates)}")

frac_idx = int(round(0.70 * (len(all_dates) - 1)))
raw_cut = pd.Timestamp(all_dates[frac_idx]).normalize()
# Snap to month-end session on or before raw_cut.
month_sessions = all_dates[all_dates <= raw_cut]
# last session of that month
cut_month = raw_cut.to_period("M")
in_month = month_sessions[month_sessions.to_period("M") == cut_month]
RESEARCH_IS_END = pd.Timestamp(in_month.max() if len(in_month) else month_sessions.max()).normalize()
PANEL_END = pd.Timestamp(all_dates.max()).normalize()
PAPER_START = (PANEL_END + pd.Timedelta(days=1)).normalize()

cuts = {
    "RESEARCH_IS_END": str(RESEARCH_IS_END.date()),
    "PANEL_START": str(pd.Timestamp(all_dates.min()).date()),
    "PANEL_END": str(PANEL_END.date()),
    "PAPER_START": str(PAPER_START.date()),
    "RESEARCH_IS_FRACTION": 0.70,
    "yahoo_drop_rate": DROP_RATE,
    "gics_miss_rate": GICS_MISS_RATE,
    "adv_drop_rate": ADV_DROP_RATE,
    "adv_floor_usd": ADV_FLOOR_USD,
    "n_union": n_union,
    "n_priced": n_ok,
    "n_tradable": len(TRADABLE),
    "dropped_tickers": DROPPED_TICKERS,
}
cuts_path = os.path.join(NOTEBOOK_DIR, "s2_cuts.json")
with open(cuts_path, "w", encoding="utf-8") as f:
    json.dump(cuts, f, indent=2)
    f.write("\n")
print("wrote", cuts_path)
print(json.dumps({k: cuts[k] for k in cuts if k != "dropped_tickers"}, indent=2))


## 6. Name panel (train / full)


In [ ]:
# Expand membership to a daily in_index flag via asof merge per ticker would be heavy;
# store sparse membership + OHLCV long panel for tradable names.
rows = []
gics_map = gics.set_index("ticker")
for t in TRADABLE:
    fr = ohlc_by_ticker[t].copy()
    fr = fr.reset_index()
    fr["ticker"] = t
    fr["sector"] = sector_of.get(t, "")
    fr["industry"] = industry_of.get(t, "")
    # ADV trailing on each row (causal).
    dollar = fr["close"] * fr["volume"]
    fr["adv_20"] = dollar.rolling(ADV_WINDOW, min_periods=max(5, ADV_WINDOW // 2)).mean()
    rows.append(fr)

name_full = pd.concat(rows, ignore_index=True)
name_full["date"] = pd.to_datetime(name_full["date"]).dt.normalize()
# in_index: true if ticker appears in any snapshot on or before date (approx via membership table)
mem = membership.copy()
mem["date"] = pd.to_datetime(mem["date"]).dt.normalize()
# For each (date, ticker) in name panel, asof-backward membership presence:
# simpler: ticker is in_index on date if any snapshot <= date listed it and no need for exit —
# use latest snapshot <= date contains ticker.
snap_sorted = snapshots.sort_values("date")
snap_sorted["date"] = pd.to_datetime(snap_sorted["date"]).dt.normalize()

# Efficient in_index via snapshot lists
snap_lists = []
for _, row in snap_sorted.iterrows():
    tickers = {t.strip().upper() for t in str(row["tickers"]).split(",") if t.strip()}
    snap_lists.append((pd.Timestamp(row["date"]).normalize(), tickers))

unique_dates = np.array(sorted(name_full["date"].unique()))
# For each unique date, find latest snap
snap_dates = np.array([s[0].to_datetime64() for s in snap_lists])
snap_sets = [s[1] for s in snap_lists]
in_index_by_date: dict[pd.Timestamp, set[str]] = {}
j = -1
for d in unique_dates:
    d_ts = pd.Timestamp(d).normalize()
    while j + 1 < len(snap_dates) and pd.Timestamp(snap_dates[j + 1]) <= d_ts:
        j += 1
    in_index_by_date[d_ts] = snap_sets[j] if j >= 0 else set()

name_full["in_index"] = [
    str(t) in in_index_by_date[pd.Timestamp(d).normalize()]
    for d, t in zip(name_full["date"], name_full["ticker"])
]

name_train = name_full.loc[name_full["date"] <= RESEARCH_IS_END].copy()
train_path = os.path.join(NOTEBOOK_DIR, "s2_name_panel_train.parquet")
full_path = os.path.join(NOTEBOOK_DIR, "s2_name_panel_full.parquet")
name_train.to_parquet(train_path, index=False)
name_full.to_parquet(full_path, index=False)
print("name train", name_train.shape, "full", name_full.shape)
print("wrote", train_path)
print("wrote", full_path)


## 7. Intra-sector EG screen → `s2_candidates_1d.csv`


In [ ]:
# Candidates: pairs inside the same sector among tradable names that are in_index at RESEARCH_IS_END.
_, members_at_cut = constituents_on_or_before(RESEARCH_IS_END)
members_at_cut = {t.strip().upper() for t in members_at_cut}
eligible = [t for t in TRADABLE if t in members_at_cut]
by_sector: dict[str, list[str]] = {}
for t in eligible:
    sec = sector_of.get(t, "")
    if not sec:
        continue
    by_sector.setdefault(sec, []).append(t)

candidate_pairs: list[tuple[str, str]] = []
for sec, names in by_sector.items():
    names = sorted(names)
    for a, b in combinations(names, 2):
        candidate_pairs.append((a, b))

print(f"sectors={len(by_sector)}  eligible_names={len(eligible)}  "
      f"intra-sector pairs={len(candidate_pairs)}")

screen = screen_pair_cointegration(
    {t: closes[t] for t in eligible if t in closes},
    candidate_pairs,
    is_end=RESEARCH_IS_END,
    ols_window=OLS_WINDOW,
    pvalue_threshold=COINT_PVALUE_THRESHOLD,
)
screen["sector_y"] = screen["ticker_y"].map(sector_of)
screen["sector_x"] = screen["ticker_x"].map(sector_of)
screen["industry_y"] = screen["ticker_y"].map(industry_of)
screen["industry_x"] = screen["ticker_x"].map(industry_of)
screen["same_industry"] = screen["industry_y"] == screen["industry_x"]
screen["same_sector"] = screen["sector_y"] == screen["sector_x"]

cand_path = os.path.join(NOTEBOOK_DIR, f"s2_candidates_{BAR}.csv")
screen.to_csv(cand_path, index=False)
n_pass = int((screen["eligible"] & (screen["pvalue"] < COINT_PVALUE_THRESHOLD)).sum())
print(f"wrote {cand_path}  rows={len(screen)}  EG passers={n_pass}")
screen.head()


## 8. Pair-panel union (freeze books) / Pass B rotate schedules


In [ ]:
def _top_pairs(df: pd.DataFrame, mask: pd.Series, n: int) -> list[tuple[str, str]]:
    sub = df.loc[mask & df["eligible"] & df["pvalue"].notna()].copy()
    sub = sub.sort_values("pvalue", kind="mergesort").head(n)
    return [(str(r.ticker_y), str(r.ticker_x)) for r in sub.itertuples(index=False)]


if BOOK_SCOPE == "freeze_only":
    # Union of top passers under both granularities.
    sector_mask = screen["same_sector"].astype(bool)
    industry_mask = screen["same_industry"].astype(bool)
    union_pairs = []
    seen = set()
    for pair in _top_pairs(screen, sector_mask, PANEL_UNION_BUFFER) + _top_pairs(
        screen, industry_mask, PANEL_UNION_BUFFER
    ):
        key = "|".join(sorted(pair))
        if key in seen:
            continue
        seen.add(key)
        union_pairs.append(pair)
    print(f"freeze union pairs={len(union_pairs)}")

    panel = build_pair_panel(
        ohlc_by_ticker,
        union_pairs,
        ols_window=OLS_WINDOW,
        z_window=Z_WINDOW,
        hl_window=HL_WINDOW,
        include_adf_pvalue=True,
        include_variance_jump=False,
        hedge="ols",
    )
    panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
    train = panel.loc[panel["date"] <= RESEARCH_IS_END].copy()
    full = panel.copy()
    train_p = os.path.join(NOTEBOOK_DIR, f"s2_panel_{BAR}_train.parquet")
    full_p = os.path.join(NOTEBOOK_DIR, f"s2_panel_{BAR}_full.parquet")
    train.to_parquet(train_p, index=False)
    full.to_parquet(full_p, index=False)
    print("wrote", train_p, train.shape)
    print("wrote", full_p, full.shape)
    print(NEXT)

else:
    # Pass B: require POOL_STAR + FDR_STAR; write rotate schedules; rebuild union panel.
    stack = load_star_stack(STAR_PATH)
    require_star("POOL_STAR", stack.get("POOL_STAR"))
    require_star("FDR_STAR", stack.get("FDR_STAR"))
    pool_star = str(stack["POOL_STAR"])
    fdr_star = str(stack["FDR_STAR"])

    from backtest.s2_coint.research import apply_bh_fdr

    cand = pd.read_csv(os.path.join(NOTEBOOK_DIR, f"s2_candidates_{BAR}.csv"))
    if pool_star == "industry":
        base = cand.loc[cand["same_industry"].astype(bool)].copy()
    else:
        base = cand.loc[cand["same_sector"].astype(bool)].copy()
    if fdr_star == "bh_fdr":
        base = apply_bh_fdr(base, alpha=0.05)
        base = base.loc[base["fdr_pass"]].copy()
    else:
        base = base.loc[base["eligible"] & (base["pvalue"] < COINT_PVALUE_THRESHOLD)].copy()

    # Universe of tickers still needed
    tickers = sorted(set(base["ticker_y"].astype(str)) | set(base["ticker_x"].astype(str)))
    close_map = {t: closes[t] for t in tickers if t in closes}
    ohlc_map = {t: ohlc_by_ticker[t] for t in tickers if t in ohlc_by_ticker}

    # Build candidate list from base pair orientations
    cand_pairs = [
        (str(r.ticker_y), str(r.ticker_x)) for r in base.itertuples(index=False)
    ]
    # Also allow unordered re-screen: use sorted unique slots from base
    pool_pairs = list({tuple(sorted(p)): p for p in cand_pairs}.values())

    dates = sorted({d for fr in ohlc_map.values() for d in fr.index})
    dates = [d for d in dates if d <= PANEL_END]
    q_dates = quarter_end_rebalance_dates(dates, end=RESEARCH_IS_END)
    s_dates = semiannual_rebalance_dates(dates, end=RESEARCH_IS_END)

    reb_rows = []
    union_slots: set[tuple[str, str]] = set()
    # Include freeze passers in union
    for p in pool_pairs[:PANEL_UNION_BUFFER]:
        union_slots.add((p[0], p[1]))

    for schedule_name, rebals in (("quarterly", q_dates), ("12m6m", s_dates)):
        for L in LOOKBACKS:
            for t in rebals:
                screen_t = screen_pair_cointegration(
                    close_map,
                    pool_pairs,
                    is_end=t,
                    ols_window=OLS_WINDOW,
                    pvalue_threshold=COINT_PVALUE_THRESHOLD,
                    lookback_bars=L,
                )
                if fdr_star == "bh_fdr":
                    screen_t = apply_bh_fdr(screen_t, alpha=0.05)
                    chosen = screen_t.loc[screen_t["fdr_pass"]].copy()
                else:
                    chosen = screen_t.loc[
                        screen_t["eligible"]
                        & (screen_t["pvalue"] < COINT_PVALUE_THRESHOLD)
                    ].copy()
                chosen = chosen.sort_values("pvalue", kind="mergesort").head(50)
                # next session after t
                later = [d for d in dates if d > t]
                if not later:
                    continue
                effective = later[0]
                for rank, row in enumerate(chosen.itertuples(index=False), start=1):
                    reb_rows.append(
                        {
                            "rebalance_date": t,
                            "effective_date": effective,
                            "pair_id": str(row.pair_id),
                            "ticker_y": str(row.ticker_y),
                            "ticker_x": str(row.ticker_x),
                            "pvalue": float(row.pvalue),
                            "rank": rank,
                            "lookback": L,
                            "schedule": schedule_name,
                            "pool": pool_star,
                        }
                    )
                    union_slots.add((str(row.ticker_y), str(row.ticker_x)))

    reb = pd.DataFrame(reb_rows)
    reb_path = os.path.join(NOTEBOOK_DIR, f"s2_rebalance_{BAR}.csv")
    reb.to_csv(reb_path, index=False)
    print("wrote", reb_path, "rows", len(reb))

    union_pairs = sorted(union_slots)
    print("rotate union pairs", len(union_pairs))
    panel = build_pair_panel(
        ohlc_map,
        union_pairs,
        ols_window=OLS_WINDOW,
        z_window=Z_WINDOW,
        hl_window=HL_WINDOW,
        include_adf_pvalue=True,
        hedge="ols",
    )
    panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
    train = panel.loc[panel["date"] <= RESEARCH_IS_END].copy()
    train_p = os.path.join(NOTEBOOK_DIR, f"s2_panel_{BAR}_train.parquet")
    full_p = os.path.join(NOTEBOOK_DIR, f"s2_panel_{BAR}_full.parquet")
    train.to_parquet(train_p, index=False)
    panel.to_parquet(full_p, index=False)
    print("wrote", train_p, train.shape)
    print("wrote", full_p, panel.shape)
    print(NEXT)


## 9. Notes

- Pass A next step: run H-001.
- Pass B next step: run H-005.
- H-003 writes `s2_panel_1h_{train,full}.parquet` when comparing bars.
- Do not open the sealed holdout here.
